<a href="https://colab.research.google.com/github/Elijah-Oluwapelumi/Coastal-Dynamics-of-Angola-A-Remote-Sensing-and-GIS-Approach-to-Shoreline-and-Bathymetric-Mapping/blob/main/Shorline_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Automated Shoreline Extraction Using Landsat-9 and Sentinel-2A Imagery with GEE Google Colab

# 1. Install and upgrade required libraries

!pip install --upgrade xee          # Installs or updates Xee, which acts as a bridge between Google Earth Engine and Xarray. Allows Earth Engine images to be opened directly as Xarray datasets.
!pip install -U geemap              # Used for interactive mapping and drawing ROI.
!pip install pycrs                  # Required for geemap to handle shapefile CRS conversions.

import ee                           # Imports the Google Earth Engine Python API.
import geemap
import zipfile
import os
import xarray as xr                 # Handles multi-dimensional raster data (time, latitude, longitude). Ideal for climate and remote sensing analysis.
import numpy as np                  # Provides numerical operations, array creation, and matrix handling.
from scipy.ndimage import binary_erosion           # Imports morphological image-processing tools. Used to shrink water regions, helping to extract shorelines.

In [ ]:
#Authenticate and initialize Google Earth Engine

ee.Authenticate()                   # Connects Python environment with your GEE account.

ee.Initialize(                      # High-volume API improves performance for large datasets.
    project='ceremonial-orb-464610-b2',
    opt_url='https://earthengine-highvolume.googleapis.com'  # High-volume API improves performance for large datasets.
)

In [ ]:
# 3. Create interactive map and select study area

interactive_map = geemap.Map()      # Creates an interactive Leaflet-based map. Supports zooming, panning, and drawing tools.
interactive_map

In [ ]:
# =========================================================
# EXTRACT ZIP FILE
# =========================================================

zip_path = '/content/Study_Area.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/study_area')

print("ZIP file extracted successfully")

# =========================================================
# LOAD SHAPEFILE INTO EARTH ENGINE
# =========================================================

study_area = geemap.shp_to_ee(
    '/content/study_area/Study_Area.shp'
)

# Convert FeatureCollection to Geometry
study_geometry = study_area.geometry()

print("Study area loaded successfully")

# =========================================================
# STYLE: RED OUTLINE ONLY (NO FILL)
# =========================================================

study_area_style = {
    'color': 'red',
    'fillColor': '00000000',
    'width': 2
}

# =========================================================
# VISUALIZE STUDY AREA
# =========================================================

Map = geemap.Map()

Map.addLayer(
    study_area.style(**study_area_style),
    {},
    'Study Area'
)

Map.centerObject(study_geometry, 8)

Map

In [ ]:
#5. NDWI function for Landsat OLI sensors
# NDWI = (Green - NIR) / (Green + NIR)       # Defines a reusable function to compute NDWI for each Landsat image.

def calculate_landsat_ndwi(image):
    ndwi_img = image.normalizedDifference(
        ['SR_B3', 'SR_B5']
    ).rename('ndwi')

    return ndwi_img

In [ ]:
# 6. Load and process Landsat 9 imagery

landsat9_ndwi = (
    ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    .filterDate('2025-01-01', '2025-12-31')
    .filterBounds(study_geometry)
    .filter(ee.Filter.lt('CLOUD_COVER', 30))
    .map(calculate_landsat_ndwi)
    .median()    # Creates a median composite: Reduces noise, Removes outliers and Produces a clean NDWI image
    .clip(study_area)
)

landsat9_ndwi

In [ ]:
# =========================
# 7. Create Binary Water Mask
# =========================

water_mask = landsat9_ndwi.gt(0)

# =========================
# 8. Extract Shoreline
# =========================

shoreline = water_mask.subtract(
    water_mask.focal_min(1)
)

# =========================
# 9. Display Results
# =========================

interactive_map = geemap.Map()

# Add NDWI layer
interactive_map.addLayer(
    landsat9_ndwi,
    {'min': -1, 'max': 1, 'palette': ['brown', 'white', 'blue']},
    'NDWI'
)


# Add water mask
interactive_map.addLayer(
    water_mask.selfMask(),
    {'palette': ['blue']},
    'Water Mask'
)

# Add shoreline
interactive_map.addLayer(
    shoreline.selfMask(),
    {'palette': ['red']},
    'Shoreline'
)

# Center map on study area
interactive_map.centerObject(study_geometry, 7)

interactive_map

In [ ]:
# Convert shoreline raster to vector
shoreline_vector = shoreline.selfMask().reduceToVectors(
    geometry=study_geometry,
    scale=30,
    geometryType='polygon',
    reducer=ee.Reducer.countEvery(),
    maxPixels=1e13
)

In [ ]:
# Export shoreline vector to Google Drive
task = ee.batch.Export.table.toDrive(
    collection=shoreline_vector,
    description='Angola_Shoreline',
    folder='GEE_Exports',
    fileFormat='SHP'
)

task.start()

print("Shoreline export started...")

In [ ]:
# Convert study area to FeatureCollection
study_area_fc = ee.FeatureCollection([
    ee.Feature(study_geometry)
])

In [ ]:
# Export study area boundary
task2 = ee.batch.Export.table.toDrive(
    collection=study_area_fc,
    description='Study_Area',
    folder='GEE_Exports',
    fileFormat='SHP'
)

task2.start()

print("Study area export started...")

In [ ]:
 # =========================================================
# 13. EXPORT NDWI RASTER
# =========================================================

task_ndwi = ee.batch.Export.image.toDrive(
    image=landsat9_ndwi,
    description='NDWI_Angola',
    folder='GEE_Exports',
    fileNamePrefix='ndwi_angola',
    region=study_geometry,
    scale=30,
    maxPixels=1e13
)

task_ndwi.start()

print("NDWI export started")


In [ ]:
# =========================================================
# 14. EXPORT WATER MASK
# =========================================================

task_water = ee.batch.Export.image.toDrive(
    image=water_mask.selfMask(),
    description='WaterMask_Angola',
    folder='GEE_Exports',
    fileNamePrefix='watermask_angola',
    region=study_geometry,
    scale=30,
    maxPixels=1e13
)

task_water.start()

print("Water mask export started")


In [ ]:
# =========================================================
# 16. CHECK EXPORT STATUS
# =========================================================

ee.batch.Task.list()